# 🔬 Standalone Out-of-Distribution & Tiled Inference Evaluation
This notebook evaluates trained YOLOv11n-seg checkpoints on **unseen, full-resolution uncropped road crack photos**:
1. **Direct Resizing Evaluation**: Standard evaluation at $512 \times 512$.
2. **Gaussian-Weighted Tiled / Sliding-Window Inference**: Dividing $2000 \times 1500$ uncropped images into overlapping $512 \times 512$ patches ($25\%$ overlap, $384\text{px}$ stride) with **2D Gaussian Apodization Blending** (eliminating border artifacts and weighting center predictions).
3. **Head-to-Head Comparison**: Compares all available checkpoints (Baseline, Full KD, Mask KD, Foreground-Dilated, LayerKD, Focal, Combined).


In [ ]:
# ── Environment & Imports ──
!mkdir -p scripts configs utils distillation inference data/datasets results
!pip install -q ultralytics albumentations pycocotools opencv-python Pillow matplotlib tqdm pandas
import os, cv2, json, time, glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO


In [ ]:
%%writefile utils/__init__.py
# utils package
from utils.checkpoint import resolve_checkpoint, get_checkpoint_manifest, save_checkpoint_manifest


In [ ]:
%%writefile utils/checkpoint.py
"""
Checkpoint provenance and deterministic resolution utility.
Addresses scientific audit findings:
- Eliminates non-deterministic glob-based checkpoint collision.
- Records immutable SHA256 manifests for every trained model.
"""

import hashlib
import json
import os
import time
from pathlib import Path
from typing import Any, Dict, Optional, Union


def compute_file_sha256(file_path: Union[str, Path], chunk_size: int = 1024 * 1024) -> str:
    """
    Compute streaming SHA256 hash of a file.

    Args:
        file_path (str | Path): Path to file.
        chunk_size (int): Read chunk size in bytes (default: 1MB).

    Returns:
        str: Hex-encoded SHA256 digest.
    """
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint file not found: {path}")

    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()


def resolve_checkpoint(
    checkpoint_path: Union[str, Path],
    expected_experiment: Optional[str] = None,
    allow_search: bool = False,
) -> Path:
    """
    Deterministically resolve a checkpoint file path without fuzzy wildcards.

    Args:
        checkpoint_path (str | Path): File path or directory containing weights.
        expected_experiment (str, optional): Expected experiment name for sanity checking.
        allow_search (bool): If True and path is directory, checks standard weights/best.pt.

    Returns:
        Path: Verified checkpoint path.

    Raises:
        FileNotFoundError: If checkpoint does not exist.
        ValueError: If file is not a valid PyTorch weight file or empty.
    """
    path = Path(checkpoint_path)

    # 1. Direct file resolution
    if path.is_file():
        if path.suffix not in (".pt", ".pth", ".bin"):
            raise ValueError(f"Checkpoint file has unsupported extension '{path.suffix}': {path}")
        if path.stat().st_size == 0:
            raise ValueError(f"Checkpoint file is empty (0 bytes): {path}")
        return path.resolve()

    # 2. Directory resolution
    if path.is_dir():
        candidates = [
            path / "weights" / "best.pt",
            path / "best.pt",
            path / "weights" / "last.pt",
            path / "last.pt",
        ]
        for candidate in candidates:
            if candidate.is_file() and candidate.stat().st_size > 0:
                return candidate.resolve()

        if expected_experiment:
            named_candidates = [
                path / expected_experiment / "weights" / "best.pt",
                path / "segment" / expected_experiment / "weights" / "best.pt",
                path / "crack_distill" / expected_experiment / "weights" / "best.pt",
            ]
            for candidate in named_candidates:
                if candidate.is_file() and candidate.stat().st_size > 0:
                    return candidate.resolve()

    raise FileNotFoundError(
        f"Could not deterministically resolve checkpoint from path '{checkpoint_path}'. "
        f"Expected existing .pt file or directory containing weights/best.pt."
    )


def get_checkpoint_manifest(
    checkpoint_path: Union[str, Path],
    experiment_name: Optional[str] = None,
    seed: Optional[int] = None,
    metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Build immutable provenance record for a checkpoint file.

    Returns dict with sha256, file size, timestamp, experiment, seed, and optional metadata.
    """
    resolved_path = resolve_checkpoint(checkpoint_path)
    stat = resolved_path.stat()
    sha256 = compute_file_sha256(resolved_path)

    manifest: Dict[str, Any] = {
        "checkpoint_path": str(resolved_path),
        "filename": resolved_path.name,
        "sha256": sha256,
        "size_bytes": stat.st_size,
        "modified_time_iso": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(stat.st_mtime)),
        "experiment": experiment_name,
        "seed": seed,
        "metadata": metadata or {},
    }
    return manifest


def save_checkpoint_manifest(manifest: Dict[str, Any], output_path: Optional[Union[str, Path]] = None) -> Path:
    """
    Write checkpoint manifest to JSON file.
    Defaults to <checkpoint_dir>/<checkpoint_stem>_manifest.json.
    """
    if output_path is None:
        ckpt_p = Path(manifest["checkpoint_path"])
        output_path = ckpt_p.parent / f"{ckpt_p.stem}_manifest.json"

    out_p = Path(output_path)
    out_p.parent.mkdir(parents=True, exist_ok=True)
    with open(out_p, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    return out_p


In [ ]:
%%writefile inference/__init__.py
# inference package
from inference.tiled_inference import tiled_predict_image_gaussian, create_gaussian_weight_map


In [ ]:
%%writefile inference/tiled_inference.py
"""
CrackDistill — Gaussian-Weighted Tiled Sliding-Window Inference Engine
=====================================================================
Performs overlapping sliding-window inference on high-resolution images
(e.g., 2000x1500 full-scene asphalt pavement photographs) using 2D Gaussian
apodization blending to eliminate tiling boundary discontinuities.
"""

from pathlib import Path
import cv2
import numpy as np
import torch


def create_gaussian_weight_map(tile_size: int = 512, sigma: float = 0.35) -> np.ndarray:
    """
    Generate a 2D Gaussian window to smoothly blend overlapping inference tiles.

    Args:
        tile_size (int): Dimension of square tile (default 512).
        sigma (float): Gaussian spread parameter relative to [-1, 1] range (default 0.35).

    Returns:
        np.ndarray: Normalized 2D float32 weight array of shape (tile_size, tile_size).
    """
    ax = np.linspace(-1, 1, tile_size)
    gauss_1d = np.exp(-0.5 * (ax / sigma) ** 2)
    gauss_2d = np.outer(gauss_1d, gauss_1d).astype(np.float32)
    gauss_2d = np.maximum(gauss_2d, 0.05)  # Minimum floor to prevent edge zero division
    return gauss_2d / gauss_2d.max()


def tiled_predict_prob_map(
    model,
    img_bgr: np.ndarray,
    tile_size: int = 512,
    stride: int = 384,
    conf: float = 0.25,
    sigma: float = 0.35,
    device: str | None = None,
) -> np.ndarray:
    """
    Run overlapping sliding-window inference on full-resolution image with
    2D Gaussian apodization blending, returning continuous probability map.

    Args:
        model: Ultralytics YOLO segmentation model or callable.
        img_bgr (np.ndarray): Full-resolution input image in BGR format (H, W, 3).
        tile_size (int): Size of sliding-window tile (default 512).
        stride (int): Sliding step in pixels; overlap is tile_size - stride (default 384 = 25% overlap).
        conf (float): Detection confidence threshold for model.predict (default 0.25).
        sigma (float): 2D Gaussian apodization parameter (default 0.35).
        device (str | None): Computing device ('cuda', 'cpu', or auto-detect if None).

    Returns:
        np.ndarray: Continuous crack probability map of shape (H, W) in range [0.0, 1.0].
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    h, w = img_bgr.shape[:2]
    full_prob_map = np.zeros((h, w), dtype=np.float32)
    weight_accum_map = np.zeros((h, w), dtype=np.float32)
    weight_window = create_gaussian_weight_map(tile_size, sigma)

    # Compute step coordinates covering the entire canvas
    y_steps = list(range(0, max(1, h - tile_size + 1), stride))
    if not y_steps or y_steps[-1] + tile_size < h:
        y_steps.append(max(0, h - tile_size))

    x_steps = list(range(0, max(1, w - tile_size + 1), stride))
    if not x_steps or x_steps[-1] + tile_size < w:
        x_steps.append(max(0, w - tile_size))

    for y0 in y_steps:
        for x0 in x_steps:
            tile = img_bgr[y0 : y0 + tile_size, x0 : x0 + tile_size]
            
            # Pad tile if canvas is smaller than tile_size
            pad_h = max(0, tile_size - tile.shape[0])
            pad_w = max(0, tile_size - tile.shape[1])
            if pad_h > 0 or pad_w > 0:
                tile = cv2.copyMakeBorder(tile, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)

            results = model.predict(tile, imgsz=tile_size, conf=conf, verbose=False, device=device)
            r = results[0]

            tile_prob = np.zeros((tile_size, tile_size), dtype=np.float32)
            if r.masks is not None and len(r.masks) > 0:
                for m in r.masks.data.cpu().numpy():
                    m_resized = cv2.resize(m, (tile_size, tile_size), interpolation=cv2.INTER_LINEAR)
                    tile_prob = np.maximum(tile_prob, m_resized)

            # Unpad if tile was padded
            actual_h = min(tile_size, h - y0)
            actual_w = min(tile_size, w - x0)

            full_prob_map[y0 : y0 + actual_h, x0 : x0 + actual_w] += (
                tile_prob[:actual_h, :actual_w] * weight_window[:actual_h, :actual_w]
            )
            weight_accum_map[y0 : y0 + actual_h, x0 : x0 + actual_w] += (
                weight_window[:actual_h, :actual_w]
            )

    weight_accum_map = np.maximum(weight_accum_map, 1e-5)
    return full_prob_map / weight_accum_map


def tiled_predict_image_gaussian(
    model,
    img_bgr: np.ndarray,
    tile_size: int = 512,
    stride: int = 384,
    conf: float = 0.25,
    threshold: float = 0.35,
    sigma: float = 0.35,
    device: str | None = None,
) -> np.ndarray:
    """
    Run overlapping sliding-window inference on full-resolution image with
    2D Gaussian apodization blending, returning binary mask (0 or 1).

    Args:
        model: Ultralytics YOLO segmentation model or callable.
        img_bgr (np.ndarray): Full-resolution input image in BGR format (H, W, 3).
        tile_size (int): Size of sliding-window tile (default 512).
        stride (int): Sliding step in pixels (default 384).
        conf (float): Detection confidence threshold (default 0.25).
        threshold (float): Final binarization probability threshold (default 0.35).
        sigma (float): 2D Gaussian apodization parameter (default 0.35).
        device (str | None): Computing device ('cuda', 'cpu', or auto-detect).

    Returns:
        np.ndarray: Binary crack segmentation mask of shape (H, W) with values {0, 1}, uint8.
    """
    prob_map = tiled_predict_prob_map(
        model=model,
        img_bgr=img_bgr,
        tile_size=tile_size,
        stride=stride,
        conf=conf,
        sigma=sigma,
        device=device,
    )
    return (prob_map > threshold).astype(np.uint8)


In [ ]:
%%writefile scripts/convert_crack500_uncropped.py
#!/usr/bin/env python3
"""
Crack500 Uncropped Test/Val → YOLO seg format converter
======================================================
Converts the original uncropped validation and test sets of Crack500.
Handles EXIF orientation for images by rotating the corresponding masks.

Source directories:
  data/datasets/crack500/valdata/   ← contains {stem}.jpg and {stem}_mask.png
  data/datasets/crack500/testdata/  ← contains {stem}.jpg and {stem}_mask.png

Output (YOLO seg format):
  data/datasets/crack500_uncropped_yolo/
  ├── images/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── val/
  │   └── test/
  └── dataset.yaml
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def get_exif_rotation(img_path: Path):
    """Retrieve EXIF orientation tag from image."""
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)  # 274 is the Orientation tag
    except Exception:
        pass
    return None


def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    """Rotate mask array to match image rotation applied by cv2.imread based on EXIF."""
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int, exif_orientation: int = None) -> list[str]:
    """
    Read binary PNG mask → rotate based on EXIF → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    if exif_orientation:
        mask = rotate_mask_to_match_image(mask, exif_orientation)

    # Threshold (Crack500 masks are binary 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_dir: Path, split_name: str):
    """Process uncropped val or test split."""
    split_dir = src_dir / f"{split_name}data"
    if not split_dir.exists():
        print(f"  [Warning] Directory {split_dir} does not exist, skipping split {split_name}.")
        return 0

    dst_img_dir = dst_dir / "images" / split_name
    dst_lbl_dir = dst_dir / "labels" / split_name

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all image files (jpg/jpeg/png that do not contain '_mask')
    all_files = sorted(split_dir.iterdir())
    image_files = [
        f for f in all_files 
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and '_mask' not in f.name.lower()
        and ':Zone.Identifier' not in f.name
    ]

    converted = 0
    skipped = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find mask (stem + "_mask.png")
        mask_path = split_dir / f"{stem}_mask.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions (matches how cv2.imread auto-rotates it based on EXIF)
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Get EXIF rotation from image
        exif_orientation = get_exif_rotation(img_path)

        # Convert mask to YOLO seg labels (rotating it to match)
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h, exif_orientation)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 Uncropped splits to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/crack500",
        help="Path to crack500 root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default="data/datasets/crack500_uncropped_yolo",
        help="Output directory"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve()

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    dst.mkdir(parents=True, exist_ok=True)

    counts = {}
    for split in ["val", "test"]:
        n = process_split(src, dst, split)
        counts[split] = n

    # Write dataset_uncropped.yaml
    yaml_content = f"""# Crack500 Uncropped — YOLO seg format
# Auto-generated by convert_crack500_uncropped.py

path: {dst.resolve()}
train: images/val
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# val:   ~{counts.get('val', 0)} images (uncropped)
# test:  ~{counts.get('test', 0)} images (uncropped)
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")
    print(f"[Done] Converted uncropped splits successfully.")


if __name__ == "__main__":
    main()


In [ ]:
# ── Step 1: Link / Prepare Uncropped Dataset ──
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

uncropped_dir = Path("data/datasets/crack500_uncropped_yolo")
uncropped_dir.mkdir(parents=True, exist_ok=True)

# Link raw crack500 if uncropped yolo not already converted
for root, dirs, files in os.walk(str(input_dir)):
    root_p = Path(root)
    if "valdata" in dirs or "testdata" in dirs:
        !python scripts/convert_crack500_uncropped.py --src {root_p} --dst data/datasets/crack500_uncropped_yolo
        break

print("Uncropped validation dataset ready at data/datasets/crack500_uncropped_yolo/")


In [ ]:
# ── Step 2: Gaussian-Weighted Tiled Sliding-Window Inference Engine ──
import torch
from inference.tiled_inference import tiled_predict_image_gaussian, create_gaussian_weight_map

print("Gaussian-weighted tiled inference engine loaded from inference.tiled_inference!")


In [ ]:
# ── Step 3: Discover Checkpoints & Run Cross-Evaluation ──
from PIL import Image

def get_exif_rotation(img_path: Path):
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)
    except Exception:
        pass
    return None

def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask

def compute_dice(pred_mask, gt_mask):
    if pred_mask.shape != gt_mask.shape:
        gt_mask = cv2.resize(gt_mask.astype(np.uint8), (pred_mask.shape[1], pred_mask.shape[0]), interpolation=cv2.INTER_NEAREST)
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    total = pred_mask.sum() + gt_mask.sum()
    if total == 0:
        return 1.0 if intersection == 0 else 0.0
    return float(2.0 * intersection / total)

ckpts = list(Path("/kaggle/input").glob("**/best.pt")) + list(Path("runs").glob("**/best.pt"))
print(f"Discovered {len(ckpts)} checkpoints:")
for c in ckpts:
    print(f"  - {c}")

# Find ground-truth uncropped images and masks
val_img_dir = Path("data/datasets/crack500_uncropped_yolo/images/val")
all_val_imgs = sorted(list(val_img_dir.glob("*.jpg")) + list(val_img_dir.glob("*.png")))

eval_summary = {}
for ckpt in ckpts:
    name = ckpt.parent.parent.name
    print(f"\n{'='*50}\nEvaluating Checkpoint: {name}\nPath: {ckpt}\n{'='*50}")
    model = YOLO(str(ckpt))
    
    # 1. Direct Resize Val (512x512)
    res_direct = model.val(data="data/datasets/crack500_uncropped_yolo/dataset.yaml", split="val", verbose=False)
    direct_mAP50 = float(res_direct.seg.map50)
    direct_mAP50_95 = float(res_direct.seg.map)
    direct_box_mAP50 = float(res_direct.box.map50)
    
    # 2. Tiled Sliding-Window Full-Resolution Dice Evaluation
    tiled_dices = []
    direct_dices = []
    
    for img_p in tqdm(all_val_imgs[:50], desc=f"  Tiling eval ({name[:20]})", leave=False):
        img_bgr = cv2.imread(str(img_p))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]
        
        # Load ground-truth mask if available
        gt_mask_path = img_p.parent.parent.parent.parent / "crack500" / "valdata" / f"{img_p.stem}_mask.png"
        if not gt_mask_path.exists():
            # Fallback search
            gt_matches = list(Path("data").glob(f"**/{img_p.stem}_mask.png"))
            if gt_matches:
                gt_mask_path = gt_matches[0]
                
        if gt_mask_path.exists():
            gt_mask = cv2.imread(str(gt_mask_path), cv2.IMREAD_GRAYSCALE)
            if gt_mask is not None:
                exif_rot = get_exif_rotation(img_p)
                if exif_rot:
                    gt_mask = rotate_mask_to_match_image(gt_mask, exif_rot)
                if gt_mask.shape[:2] == (w, h):
                    gt_mask = cv2.rotate(gt_mask, cv2.ROTATE_90_CLOCKWISE)
                if gt_mask.shape[:2] != (h, w):
                    gt_mask = cv2.resize(gt_mask, (w, h), interpolation=cv2.INTER_NEAREST)
                gt_binary = (gt_mask > 127).astype(np.uint8)
                
                # A. Direct resize prediction
                r_dir = model.predict(img_bgr, imgsz=512, conf=0.25, verbose=False)[0]
                pred_dir = np.zeros((h, w), dtype=np.uint8)
                if r_dir.masks is not None and len(r_dir.masks) > 0:
                    for m in r_dir.masks.data.cpu().numpy():
                        m_resized = cv2.resize(m, (w, h))
                        pred_dir = np.maximum(pred_dir, (m_resized > 0.35).astype(np.uint8))
                direct_dices.append(compute_dice(pred_dir, gt_binary))
                
                # B. Gaussian Tiled sliding-window prediction
                pred_tiled = tiled_predict_image_gaussian(model, img_bgr, tile_size=512, stride=384, conf=0.25)
                tiled_dices.append(compute_dice(pred_tiled, gt_binary))
                
    mean_direct_dice = float(np.mean(direct_dices)) if direct_dices else 0.0
    mean_tiled_dice = float(np.mean(tiled_dices)) if tiled_dices else 0.0
    
    eval_summary[name] = {
        "direct_mask_mAP50": direct_mAP50,
        "direct_mask_mAP50_95": direct_mAP50_95,
        "direct_box_mAP50": direct_box_mAP50,
        "full_res_direct_dice": mean_direct_dice,
        "full_res_tiled_dice": mean_tiled_dice
    }
    
    print(f"Results for {name} :")
    print(f"  Direct Resize Mask mAP50    : {direct_mAP50:.4f}")
    print(f"  Direct Resize Mask mAP50-95 : {direct_mAP50_95:.4f}")
    print(f"  Full-Res Direct Dice        : {mean_direct_dice:.4f}")
    print(f"  Full-Res Tiled Dice         : {mean_tiled_dice:.4f}")

out_path = Path("/kaggle/working/results/ood_eval_summary.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(eval_summary, f, indent=2)
print(f"\nSaved evaluation summary to {out_path}")
